In [4]:
import os
import pandas as pd
import asyncio
import nest_asyncio
from datasets import Dataset

# Required to run nested event loops safely within Jupyter Notebooks
nest_asyncio.apply()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate

# Importing the exact metric class names to prevent ImportError
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextRecall, ContextPrecision

# Provide your raw API key string
MY_API_KEY = "<Enter your APIs>"
# Set environment memory fallback to make absolutely sure everything authenticates
os.environ["OPENAI_API_KEY"] = MY_API_KEY

print("🔄 Step 1: Initializing Evaluator LLM & Embeddings...")

# Initialize ChatOpenAI
openai_llm = ChatOpenAI(
    model="gpt-4o-mini", 
    temperature=0.0,
    openai_api_key=MY_API_KEY,
    api_key=MY_API_KEY
)
evaluator_llm = LangchainLLMWrapper(openai_llm)

# FIX: Removed the duplicate 'api_key' parameter that triggered the Pydantic ValidationError
openai_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY  # This is the exact parameter your current version expects
)
evaluator_embeddings = LangchainEmbeddingsWrapper(openai_embeddings)

print("✅ Evaluator LLM and Embeddings ready successfully with no validation blocks!")

🔄 Step 1: Initializing Evaluator LLM & Embeddings...
✅ Evaluator LLM and Embeddings ready successfully with no validation blocks!


In [2]:
print(" Step 2: Preparing the RAG Evaluation Dataset...")

# We construct a dataset simulating real production logs from an enterprise HR Bot.
# Notice that Row 1 is a perfect RAG response, while Row 2 contains a flaw.
production_logs = {
    "user_input": [
        "What is the policy for paid paternity leave?",
        "Can I use my personal car for commercial racing under the insurance policy?"
    ],
    "contexts": [
        ["Employees must complete a minimum of 12 consecutive months of service to be eligible for paid paternity leave."],
        ["Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing or speed testing."]
    ],
    "response": [
        "To qualify for paid paternity leave, you must complete at least 12 consecutive months of service.",
        "Yes, you are covered as long as you drive carefully." # <-- Defective AI response!
    ],
    "reference": [
        "Employees must complete a minimum of 12 consecutive months of service to be eligible for paid paternity leave.",
        "Our company premium insurance plan rejects any vehicle claims involving unauthorized commercial racing or speed testing."
    ]
}

# Convert our raw logs into a HuggingFace Dataset object required by RAGAS
evaluation_dataset = Dataset.from_dict(production_logs)
print("✅ Dataset successfully transformed and ready.")

 Step 2: Preparing the RAG Evaluation Dataset...
✅ Dataset successfully transformed and ready.


In [3]:
print("📊 Step 3: Running RAGAS Evaluation Pipeline across the Triad...")

# Execute the complete bulk evaluation framework over our dataset
evaluation_report = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
        LLMContextRecall(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm)
    ]
)

print("\n=========== 🏆 FINAL RAGAS LAB EVALUATION SCOREBOARD ===========\n")
# Convert results directly into a structured notebook pandas view
report_df = evaluation_report.to_pandas()

# FIX: Changed 'response_relevancy' to 'answer_relevancy' to match Pandas DataFrame output schema
report_df[['user_input', 'faithfulness', 'answer_relevancy', 'context_recall', 'context_precision']]

📊 Step 3: Running RAGAS Evaluation Pipeline across the Triad...


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]


=========== 🏆 FINAL RAGAS LAB EVALUATION SCOREBOARD ===========



,user_input,faithfulness,answer_relevancy,context_recall,context_precision
0,What is the policy for paid paternity leave?,1.0,0.835141,1.0,1.0
1,Can I use my personal car for commercial racin...,0.0,0.502562,1.0,1.0
